# Microdados ENEM 2023 — Região Nordeste

Atividade pontuada apresentada como **projeto final** do 5º módulo da pós-graduação.

## Imports

In [2]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

## Ler Dataset

Cada linha do `df` corresponde a uma pessoa inscrita no ENEM

In [ ]:
microdados_enem_df = pd.read_csv(
    filepath_or_buffer="../../data/microdados-enem-2023-nordeste.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    encoding_errors="ignore"
)

microdados_enem_df.head()

## Tratamento dos Dados

Em um novo `df`, selecionar as seguintes colunas:
- `SG_UF_PROVA` — Sigla do estado onde o inscrito realizou a prova
- `NU_NOTA_MT` — Nota da prova de Matemática

In [ ]:
notas_mt_por_estado_df = microdados_enem_df[[
    "SG_UF_PROVA",
    "NU_NOTA_MT"
]]

notas_mt_por_estado_df.head()

Remover linhas `NaN`

In [ ]:
notas_mt_por_estado_df = notas_mt_por_estado_df.dropna(axis=0, how="any")

## Questão 01

Verifique se as notas de matemática (`NU_NOTA_MT`), para cada estado, podem ser aproximada por uma *Distribuição Normal* (dica: Utilize o teste de *Shapiro-Wilk* ou *Kolmogorov-Smirnov* em uma amostra).

Definir função para o teste de *Kolmogorov-Smirnov*

In [ ]:
def aplicar_teste_ks(df: pd.DataFrame, amostra: int=5000) -> pd.DataFrame:
    resultados = {}
    siglas_uf = df["SG_UF_PROVA"].unique()

    for uf in siglas_uf:
        notas = df[df["SG_UF_PROVA"] == uf]["NU_NOTA_MT"]
        total_notas = len(notas)

        if total_notas >= 3:
            if total_notas > amostra:
                notas = notas.sample(amostra, random_state=42)

            notas_normalizadas = (notas - notas.mean()) / notas.std()
            ks_stat, p_value = stats.kstest(rvs=notas_normalizadas, cdf="norm")
            resultados[uf] = {
                "KS Stat": ks_stat,
                "p-value": p_value,
                "n": total_notas
            }
        else:
            resultados[uf] = {
                "KS Stat": None,
                "p-value": None,
                "n": total_notas
            }

    return pd.DataFrame(resultados).T

Aplicar o teste

In [ ]:
resultado_teste_ks_df = aplicar_teste_ks(notas_mt_por_estado_df)
resultado_teste_ks_df.head(n=10)

## Questão 02

Faça um comparativo entre a médias das notas de matemática (`NU_NOTA_MT`) e verifique, por estado, se houve diferença significativa entre os estados com nível de confiança de 90, 95 e 99%. Apresente os resultados em forma de tabela.

Aplicar teste ANOVA (análise de variância)

In [ ]:
siglas_uf = notas_mt_por_estado_df["SG_UF_PROVA"].unique()
grupos_de_estados = [notas_mt_por_estado_df[notas_mt_por_estado_df["SG_UF_PROVA"] == uf]["NU_NOTA_MT"] for uf in siglas_uf]
f_stat, p_nova = stats.f_oneway(*grupos_de_estados)
print(f"[ANOVA] F-statistic: {f_stat:.4f} | p-value: {p_nova:.4g}")

# [ANOVA] F-statistic: 113.1344 | p-value: 3.141e-189

Aplicar testes de *Tukey's Honestly Significant Difference*

1. Teste com 90% de confiança

In [ ]:
tukey_90 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.10 # nível de confiança
)

tukey_90_df = pd.DataFrame(
    data=tukey_90._results_table.data[1:],
    columns=tukey_90._results_table.data[0]
)

tukey_90_df.head(n=36)

2. Teste com 95% de confiança

In [ ]:
tukey_95 = tukey_90 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.05 # nível de confiança
)

tukey_95_df = pd.DataFrame(
    data=tukey_95._results_table.data[1:],
    columns=tukey_95._results_table.data[0]
)

tukey_95_df.head(n=36)

3. Teste com 99% de confiança

In [ ]:
tukey_99 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.01 # nível de confiança
)

tukey_99_df = pd.DataFrame(
    data=tukey_99._results_table.data[1:],
    columns=tukey_99._results_table.data[0]
)

tukey_99_df.head(n=36)